# Lab - Claude Code Hooks: A Rule That Cannot Break

Three tasks that give Claude Code a rule it cannot break: a hook that blocks dangerous commands before they run. A headless run in a pipeline has nobody watching - the only safe no is one enforced by code.

| Task | What you learn |
| --- | --- |
| 1 | A guard script: the event as JSON on stdin, exit 2 and a reason to block |
| 2 | Register it in .claude/settings.json, so the rule travels with the repo |
| 3 | Test it headless: the hook blocks what --allowedTools cannot |

**You do not need Claude Code or an API key.** `claude_code_run()` plays `claude -p ... --allowedTools ...`: it reads your `.claude/settings.json` and runs your guard script for real, as a subprocess with the event on standard input, before every Bash command. The commands themselves are interpreted, never executed. The files you write are exactly the files Claude Code uses at home.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check
from shopassist_lab import claude_code_run   # plays `claude -p ... --allowedTools ...`
import json
from pathlib import Path

# The project folder: a scratch file nobody wants deleted.
Path("review.json").write_text('{"status": "draft"}')
Path(".claude/hooks").mkdir(parents=True, exist_ok=True)

# A second copy of the project with no hooks at all, for comparison in Task 3.
Path("unguarded").mkdir(exist_ok=True)
(Path("unguarded") / "review.json").write_text('{"status": "draft"}')
if not (Path("unguarded") / ".env").exists():   # a fake secret, in the copy only
    (Path("unguarded") / ".env").write_text("WEBHOOK_SECRET=whsec_lab_fake_0000\n")

---

## Task 1 - A guard script that exits with 2

In [ ]:
# ============================================================
# TASK 1 - A guard script that exits with 2
# ============================================================
#
# WHAT TO DO
#   Fill in the guard's rule - what a command must not contain,
#   and the exit code that blocks it - and write the script to
#   .claude/hooks/guard.py. It reads the event from standard
#   input as JSON and takes the shell command from the tool
#   input.
#
# WHY IT MATTERS
#   Claude Code hooks are small programs that run at fixed
#   moments. A PreToolUse hook sees the tool and its input
#   before anything happens. If it exits with code 2, the call
#   is blocked, and whatever it wrote to standard error goes
#   back to Claude as the reason. A headless run in a pipeline
#   has nobody watching: the only safe no is one enforced by
#   code.
#
# WHERE TO SEE IT IN THE LECTURE
#   "First, the guard itself".
#
# HOW TO DO IT
#   Three blanks:
#
#       "rm "   a delete: the remove command followed by a space
#       ".env"  any mention of the file where the key lives
#       2       the exit code that blocks the tool call
# ============================================================

try:
    # BEGIN SOLUTION
    BLOCKED = ["rm ", ".env"]
    BLOCK_EXIT = 2
    # SCAFFOLD: BLOCKED = [..., ...]   # "rm ", ".env"
    # SCAFFOLD: BLOCK_EXIT = ...   # 2 - the code that blocks
    # END SOLUTION: replace each ... below with the value named beside it

    if ... in BLOCKED or BLOCK_EXIT is ...:   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
    GUARD = f"""import json
import sys

event = json.load(sys.stdin)
command = event.get("tool_input", {{}}).get("command", "")
if any(part in command for part in {BLOCKED!r}):
    print("Blocked by policy: no deletes, no access to .env", file=sys.stderr)
    sys.exit({BLOCK_EXIT})
"""
    path = ".claude/hooks/guard.py"
    Path(path).write_text(GUARD)
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(GUARD)

check("hook_script", path=path)

---

## Task 2 - Register it in the project's settings

In [ ]:
# ============================================================
# TASK 2 - Register it in the project's settings
# ============================================================
#
# WHAT TO DO
#   Write .claude/settings.json: under hooks, PreToolUse, a
#   matcher on Bash, and the command that runs the guard.
#
# WHY IT MATTERS
#   Claude Code reads this file from the project folder. So the
#   rule is not on one laptop - it travels with the repository,
#   and every teammate and every pipeline that runs Claude Code
#   here gets it. The matcher picks the tool the hook watches;
#   the command is how Claude Code starts the script.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then the settings file".
#
# HOW TO DO IT
#   Three blanks:
#
#       "PreToolUse"                       before the tool runs
#       "Bash"                             the tool to watch
#       "python3 .claude/hooks/guard.py"   how to start the guard
# ============================================================

try:
    # BEGIN SOLUTION
    SETTINGS = {
        "hooks": {
            "PreToolUse": [
                {"matcher": "Bash",
                 "hooks": [{"type": "command", "command": "python3 .claude/hooks/guard.py"}]},
            ]
        }
    }
    # SCAFFOLD: SETTINGS = {
    # SCAFFOLD:     "hooks": {
    # SCAFFOLD:         ...: [   # "PreToolUse"
    # SCAFFOLD:             {"matcher": ...,   # "Bash"
    # SCAFFOLD:              "hooks": [{"type": "command", "command": ...}]},   # "python3 .claude/hooks/guard.py"
    # SCAFFOLD:         ]
    # SCAFFOLD:     }
    # SCAFFOLD: }
    # END SOLUTION: replace each ... below with the value named beside it
    settings_path = ".claude/settings.json"
    Path(settings_path).write_text(json.dumps(SETTINGS, indent=2))
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(Path(settings_path).read_text())

check("hook_settings", settings_path=settings_path)

---

## Task 3 - Test it headless

In [ ]:
# ============================================================
# TASK 3 - Test it headless
# ============================================================
#
# WHAT TO DO
#   Run Claude Code headless with a mixed task - show the first
#   line of .env, then delete the review file - with Bash
#   allowed. Then run the same task in the unguarded copy of the
#   project, which has no hooks.
#
# WHY IT MATTERS
#   It is tempting to rely on the allowed tools list. Allowed
#   tools decide which tools exist: Bash, yes or no. They cannot
#   say "Bash yes, but not this command". The hook reads the
#   command itself and judges it. Test it the way the pipeline
#   will run it: headless, with nobody watching.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the test".
#
# HOW TO DO IT
#   Three blanks:
#
#       ["Bash"]        the allowed tools
#       "unguarded"     the copy of the project without hooks
#       "review.json"   the file that must still be there
# ============================================================

PROMPT = ("Show the first line of .env with head, then delete review.json with rm. "
          "Report what happened.")
try:
    # BEGIN SOLUTION
    run = claude_code_run(PROMPT, allowed_tools=["Bash"])
    print(run["result"], "\n")
    open_run = claude_code_run(PROMPT, allowed_tools=["Bash"], cwd="unguarded")
    print(open_run["result"], "\n")
    print("review.json still there:", Path("review.json").exists())
    # SCAFFOLD: run = claude_code_run(PROMPT, allowed_tools=...)   # ["Bash"]
    # SCAFFOLD: print(run["result"], "\n")
    # SCAFFOLD: open_run = claude_code_run(PROMPT, allowed_tools=["Bash"], cwd=...)   # "unguarded"
    # SCAFFOLD: print(open_run["result"], "\n")
    # SCAFFOLD: print("review.json still there:", Path(...).exists())   # "review.json"
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("headless_test", run=run, open_run=open_run)

---

## Done

For a line Claude must never cross, write a PreToolUse hook that exits with 2, register it in settings.json inside the project, and test it headless - the way the pipeline will run it.